# June 24–30 TA-LSTM Data Builder (2019–2025)

이 노트북은 **LSTM 효과 검증용 6월 데이터셋**을 한 번에 준비합니다.

- 기간: **2019~2025년, 매년 6월 24일~30일**
- GK-2A: **12:00~14:00 KST, 10분 간격 13시점, 16채널**
- ASOS: 같은 날짜 **14:00 TA/HM 라벨**
- 기존 `SME_DATA` 파이프라인의 downloader / builder를 그대로 사용
- 중간 종료 후 재실행 시 이미 받은 파일은 재사용
- 8월 데이터와 충돌하지 않도록 **6월 전용 Drive 폴더** 사용
- 최종 산출물:
  - 원본 통합 `shortterm_long_2019to2025.csv`
  - 14시 라벨 `shortterm_labels_1400_2019to2025.csv`
  - **TA-LSTM 학습용 clean long CSV**
  - 품질검사 summary


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 0. 설정

In [ ]:
from pathlib import Path

YEARS = list(range(2019, 2026))
START_MMDD = '06-24'
END_MMDD = '06-30'
START_TIME = '12:00'
END_TIME = '14:00'
STEP_MINUTES = 10
ALLOW_INCOMPLETE_BUILD = True
RESUME_BUILD = True

BRANCH = 'agent/shortterm-12to14-pipeline'
REPO_URL = 'https://github.com/tswaincae1221/SME_DATA.git'
REPO_DIR = Path('/content/SME_DATA')
OUTPUT_ROOT = Path('/content/drive/MyDrive/SME_DATA/processed_station_features/shortterm_12to14_june_0624_0630')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

EXPECTED_DAYS_PER_YEAR = 7
EXPECTED_TIMES = [1200,1210,1220,1230,1240,1250,1300,1310,1320,1330,1340,1350,1400]
EXPECTED_CHANNELS = ['VI004','VI005','VI006','VI008','NR013','NR016','SW038','WV063','WV069','WV073','IR087','IR096','IR105','IR112','IR123','IR133']
print('OUTPUT_ROOT:', OUTPUT_ROOT)


## 1. GitHub 저장소 준비 + 패키지 설치

In [ ]:
import os, subprocess, sys
if not REPO_DIR.exists():
    subprocess.run(['git','clone','-b',BRANCH,REPO_URL,str(REPO_DIR)], check=True)
else:
    os.chdir(REPO_DIR)
    subprocess.run(['git','fetch','origin'], check=True)
    subprocess.run(['git','checkout',BRANCH], check=True)
    subprocess.run(['git','pull','origin',BRANCH], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','requests','pandas','numpy','xarray','h5netcdf','netCDF4','pyyaml','pyarrow'], check=True)


## 2. API 환경변수 확인

실행 전에 Colab 세션에 필요한 API 환경변수를 안전한 방식으로 설정한 뒤 아래 셀을 실행하세요. 키 값은 노트북이나 GitHub에 저장하지 않습니다.

In [ ]:
import os
if not os.environ.get('KMA_API_KEY'):
    raise RuntimeError('필요한 API 환경변수를 먼저 설정하세요.')
print('API 환경변수 확인 완료')

# Phase 1 — GK-2A + 14:00 ASOS 다운로드


In [ ]:
from collections import deque
def run_streaming(cmd, cwd, tail_lines=100):
    print('$',' '.join(map(str,cmd)),flush=True)
    tail=deque(maxlen=tail_lines)
    p=subprocess.Popen(list(map(str,cmd)),cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in p.stdout:
        print(line,end=''); tail.append(line)
    rc=p.wait()
    if rc!=0: raise RuntimeError(f'명령 실패 (exit code={rc})\n'+''.join(tail))
collect_cmd=[sys.executable,'-u','scripts/shortterm_multiyear_phased.py','--phase','collect','--years',*[str(y) for y in YEARS],'--start-mmdd',START_MMDD,'--end-mmdd',END_MMDD,'--start-time',START_TIME,'--end-time',END_TIME,'--step-minutes',str(STEP_MINUTES),'--output-root',str(OUTPUT_ROOT)]
run_streaming(collect_cmd,REPO_DIR)

## 3. 다운로드 상태 확인

In [ ]:
status_cmd=[sys.executable,'-u','scripts/shortterm_multiyear_phased.py','--phase','status','--years',*[str(y) for y in YEARS],'--start-mmdd',START_MMDD,'--end-mmdd',END_MMDD,'--start-time',START_TIME,'--end-time',END_TIME,'--step-minutes',str(STEP_MINUTES),'--output-root',str(OUTPUT_ROOT)]
run_streaming(status_cmd,REPO_DIR)

# Phase 2 — 위성 픽셀 추출 + 14:00 TA/HM 병합


In [ ]:
build_cmd=[sys.executable,'-u','scripts/shortterm_multiyear_phased.py','--phase','build','--years',*[str(y) for y in YEARS],'--start-mmdd',START_MMDD,'--end-mmdd',END_MMDD,'--start-time',START_TIME,'--end-time',END_TIME,'--step-minutes',str(STEP_MINUTES),'--output-root',str(OUTPUT_ROOT)]
if ALLOW_INCOMPLETE_BUILD: build_cmd.append('--allow-incomplete')
if RESUME_BUILD: build_cmd.append('--resume-build')
run_streaming(build_cmd,REPO_DIR)

## 5. 최종 LSTM 학습용 clean CSV 생성 + 품질검사

In [ ]:
import json, numpy as np, pandas as pd
COMBINED_DIR=OUTPUT_ROOT/'datasets'/'combined'
LONG_PATH=COMBINED_DIR/'shortterm_long_2019to2025.csv'
WIDE_PATH=COMBINED_DIR/'shortterm_wide_2019to2025.csv'
LABEL_PATH=COMBINED_DIR/'shortterm_labels_1400_2019to2025.csv'
for path in [LONG_PATH,WIDE_PATH,LABEL_PATH]:
    if not path.exists(): raise FileNotFoundError(path)
long_df=pd.read_csv(LONG_PATH)
required=['Date','TimeKST','STN_ID','LAT','LON','ALT','TA',*EXPECTED_CHANNELS]
missing=[c for c in required if c not in long_df.columns]
if missing: raise ValueError(f'필수 컬럼 누락: {missing}')
long_df['TimeKST']=pd.to_numeric(long_df['TimeKST'],errors='raise').astype(int)
seq=long_df.groupby(['Date','STN_ID'])['TimeKST'].agg(['size',lambda s:tuple(sorted(s.tolist()))]); seq.columns=['n_rows','times']
valid_time=seq[(seq.n_rows==len(EXPECTED_TIMES))&(seq.times==tuple(EXPECTED_TIMES))].reset_index()[['Date','STN_ID']]
label=long_df[long_df.TimeKST==1400][['Date','STN_ID','TA']]
valid_label=label[label.TA.notna()][['Date','STN_ID']].drop_duplicates()
keys=valid_time.merge(valid_label,on=['Date','STN_ID'])
clean=long_df.merge(keys,on=['Date','STN_ID'],validate='m:1').sort_values(['Date','STN_ID','TimeKST']).reset_index(drop=True)
CLEAN_CSV=COMBINED_DIR/'june_0624_0630_ta_lstm_long_2019to2025_clean.csv'
CLEAN_PARQUET=COMBINED_DIR/'june_0624_0630_ta_lstm_long_2019to2025_clean.parquet'
CLEAN_LABELS=COMBINED_DIR/'june_0624_0630_ta_labels_1400_2019to2025_clean.csv'
clean.to_csv(CLEAN_CSV,index=False); clean.to_parquet(CLEAN_PARQUET,index=False)
clean[clean.TimeKST==1400][['Date','STN_ID','TA']].drop_duplicates(['Date','STN_ID']).to_csv(CLEAN_LABELS,index=False)
sat=clean[EXPECTED_CHANNELS]; miss=int(sat.isna().sum().sum()); total=int(sat.shape[0]*sat.shape[1])
summary={'clean_sequences':int(len(keys)),'clean_rows':int(len(clean)),'unique_stations':int(clean.STN_ID.nunique()),'satellite_missing_fraction':float(miss/total if total else np.nan),'clean_csv':str(CLEAN_CSV)}
SUMMARY_JSON=COMBINED_DIR/'june_0624_0630_ta_lstm_data_summary.json'; SUMMARY_JSON.write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(summary,ensure_ascii=False,indent=2))


## 6. TA-LSTM 입력 파일

최종 입력은 `june_0624_0630_ta_lstm_long_2019to2025_clean.csv` 입니다.

In [ ]:
print('TA-LSTM input:', CLEAN_CSV)
print('14:00 TA labels:', CLEAN_LABELS)
print('summary:', SUMMARY_JSON)